<a href="https://colab.research.google.com/github/Githaigaa/KnowledgeEngineering/blob/main/Autonomous_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Autonomous Trailer Navigation Agent**

****

*Extends the Smart Traffic Light Navigation Agent: instead of reacting to one percept (light colour), the trailer reasons over several percepts (hitch, load, obstacle, light, weather, road) using a rule base with priorities, derived facts and explanations.*

**Step 1: Identify the Task**

The task is to help an autonomous trailer decide what to do next, based on what its sensors report.

**Goal:** Make safe driving decisions with minimal risk.

**Possible actions:** `EMERGENCY_BRAKE`, `STOP`, `STOP_AND_CHECK`, `SLOW`, `PROCEED_WITH_CAUTION`, `PROCEED`

**Step 2: Assemble Relevant Knowledge**

The system needs to know:
*   Whether the hitch is locked and the load is secured (a trailer must never move otherwise)
*   How far away the nearest obstacle is, and how far the trailer needs to stop
*   What the traffic light shows
*   The weather and the road surface, since they change grip (friction) and visibility
*   What each combination means for the action and a safe speed cap

**Fetched live weather from the free **Open-Meteo** API (`https://api.open-meteo.com/v1/forecast`, no API key, public weather data). If the API is unreachable, the agent falls back to a manually entered weather value, so the notebook always runs.

Friction coefficients and reaction time below are **engineering assumptions** for a teaching model, not certified values. Replace them with figures from your vehicle's specifications and local road-safety regulations.

In [ ]:
import json
import urllib.request

SITE_LAT, SITE_LON = -1.3061, 36.8164   # Location: Madaraka, Nairobi

def fetch_weather(lat=SITE_LAT, lon=SITE_LON, timeout=5):
    """Query Open-Meteo for current conditions and map them to our vocabulary.
    Returns (weather, source). Falls back to 'Clear' with source 'fallback' on any error."""

    url = (f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}"
           "&current=precipitation,weather_code&hourly=visibility&forecast_days=1")
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            data = json.load(r)
        cur = data["current"]
        vis = data["hourly"]["visibility"][0]          # metres
        if cur["weather_code"] >= 95:   return "Storm", "open-meteo"
        if vis is not None and vis < 1000: return "Fog", "open-meteo"
        if cur["precipitation"] > 0:    return "Rain", "open-meteo"
        return "Clear", "open-meteo"
    except Exception as e:
        return "Clear", f"fallback ({type(e).__name__})"

live_weather, weather_source = fetch_weather()
print("Weather:", live_weather, "| source:", weather_source)

Weather: Clear | source: open-meteo


**Step 3: Define the Vocabulary**
*   Predicates (represented in Python as keys of the `facts` dictionary): `HitchLocked(bool)`, `LoadSecured(bool)`, `Loaded(bool)`, `ObstacleDistance(metres)`, `Speed(km/h)`, `Light(colour)`, `Weather(w)`, `Road(r)`
*   Derived predicates (inferred, not sensed): `Friction(mu)`, `StoppingDistance(metres)`
*   Constants:
    *   Lights: Red, Yellow, Green, Flashing, None
    *   Weather: Clear, Rain, Fog, Storm
    *   Road: Paved, Dirt, Muddy
    *   Actions: EMERGENCY_BRAKE, STOP, STOP_AND_CHECK, SLOW, PROCEED_WITH_CAUTION, PROCEED

**Step 4: Encode General Knowledge about the Domain** `[3 marks]`

In [ ]:
# Vocabulary (constants)
LIGHTS  = {"Red", "Yellow", "Green", "Flashing", "None"}
WEATHER = {"Clear", "Rain", "Fog", "Storm"}
ROADS   = {"Paved", "Dirt", "Muddy"}

#General knowledge: grip depends on road and weather (assumed teaching values)
BASE_FRICTION = {"Paved": 0.8, "Dirt": 0.5, "Muddy": 0.3}
WEATHER_FACTOR = {"Clear": 1.0, "Rain": 0.7, "Fog": 0.95, "Storm": 0.6}
REACTION_TIME_S = 1.5        # sensing + actuation delay
LOADED_BRAKE_PENALTY = 1.25  # assumption: a loaded trailer needs ~25% more distance.
G = 9.81

#Derived knowledge (forward-chaining step)
def derive(facts):
    # Add inferred facts: Friction and StoppingDistance.
    f = dict(facts)
    f["Friction"] = BASE_FRICTION[f["Road"]] * WEATHER_FACTOR[f["Weather"]]
    v = f["Speed"] / 3.6                                    # m/s
    d = v * REACTION_TIME_S + v**2 / (2 * f["Friction"] * G)
    if f["Loaded"]:
        d *= LOADED_BRAKE_PENALTY
    f["StoppingDistance"] = round(d, 1)
    return f

#Rule base: IF condition THEN action (higher priority wins; speed caps combine by min)
rules = [
    # name, priority, condition(facts), action, speed_cap_kmh, explanation
    {"name": "R1_obstacle_too_close", "priority": 100,
     "if": lambda f: f["ObstacleDistance"] is not None and f["ObstacleDistance"] < f["StoppingDistance"],
     "action": "EMERGENCY_BRAKE", "cap": 0,
     "why": "Obstacle is closer than the distance needed to stop."},

    {"name": "R2_hitch_unlocked", "priority": 95,
     "if": lambda f: not f["HitchLocked"],
     "action": "STOP", "cap": 0, "why": "Hitch is not locked; trailer must not move."},

    {"name": "R3_load_unsecured", "priority": 90,
     "if": lambda f: not f["LoadSecured"],
     "action": "STOP", "cap": 0, "why": "Load is not secured."},

    {"name": "R4_red_light", "priority": 80,
     "if": lambda f: f["Light"] == "Red",
     "action": "STOP", "cap": 0, "why": "Red light: stop."},

    {"name": "R5_flashing_light", "priority": 75,
     "if": lambda f: f["Light"] == "Flashing",
     "action": "STOP_AND_CHECK", "cap": 0, "why": "Flashing signal: stop and check the intersection."},

    {"name": "R6_fog_or_storm", "priority": 70,
     "if": lambda f: f["Weather"] in {"Fog", "Storm"},
     "action": "SLOW", "cap": 30, "why": "Poor visibility or severe weather."},

    {"name": "R7_yellow_light", "priority": 60,
     "if": lambda f: f["Light"] == "Yellow",
     "action": "SLOW", "cap": 20, "why": "Yellow light: slow down and prepare to stop."},

    {"name": "R8_muddy_road", "priority": 55,
     "if": lambda f: f["Road"] == "Muddy",
     "action": "SLOW", "cap": 20, "why": "Muddy surface: low grip."},

    {"name": "R9_rain_or_dirt", "priority": 50,
     "if": lambda f: f["Weather"] == "Rain" or f["Road"] == "Dirt",
     "action": "PROCEED_WITH_CAUTION", "cap": 40, "why": "Reduced grip: cap speed."},

    {"name": "R10_all_clear", "priority": 40,
     "if": lambda f: f["Light"] in {"Green", "None"} and f["Weather"] == "Clear" and f["Road"] == "Paved",
     "action": "PROCEED", "cap": 60, "why": "Conditions are clear."},

    {"name": "R11_default", "priority": 1, "default": True,
     "if": lambda f: True,
     "action": "PROCEED_WITH_CAUTION", "cap": 30, "why": "No specific rule applies: default to caution."},
]

def show_kb():
    print("\n--- Current Knowledge Base (rules) ---")
    for r in sorted(rules, key=lambda r: -r["priority"]):
        print(f"[{r['priority']:3}] {r['name']:24} -> {r['action']:22} cap={r['cap']:>2} km/h | {r['why']}")
    print("--------------------------------------")

show_kb()


--- Current Knowledge Base (rules) ---
[100] R1_obstacle_too_close    -> EMERGENCY_BRAKE        cap= 0 km/h | Obstacle is closer than the distance needed to stop.
[ 95] R2_hitch_unlocked        -> STOP                   cap= 0 km/h | Hitch is not locked; trailer must not move.
[ 90] R3_load_unsecured        -> STOP                   cap= 0 km/h | Load is not secured.
[ 80] R4_red_light             -> STOP                   cap= 0 km/h | Red light: stop.
[ 75] R5_flashing_light        -> STOP_AND_CHECK         cap= 0 km/h | Flashing signal: stop and check the intersection.
[ 70] R6_fog_or_storm          -> SLOW                   cap=30 km/h | Poor visibility or severe weather.
[ 60] R7_yellow_light          -> SLOW                   cap=20 km/h | Yellow light: slow down and prepare to stop.
[ 55] R8_muddy_road            -> SLOW                   cap=20 km/h | Muddy surface: low grip.
[ 50] R9_rain_or_dirt          -> PROCEED_WITH_CAUTION   cap=40 km/h | Reduced grip: cap speed.
[ 40] 

**Step 5: Add New Facts to the Knowledge Base**

In [ ]:
def add_rule(name, priority, condition, action, cap, why):
    #Add a new rule to the KB without user input.
    rules.append({"name": name, "priority": priority, "if": condition,
                  "action": action, "cap": cap, "why": why})
    print(f"Added rule: {name} -> {action}")

# Example 1: a new light colour the trailer should treat like Red
add_rule("R12_white_light", 85, lambda f: f["Light"] == "White",
         "STOP", 0, "White light (e.g. railway crossing signal): stop immediately.")

# Example 2: a new road fact -> new vocabulary entry
LIGHTS.add("White")
BASE_FRICTION["Gravel"] = 0.6
ROADS.add("Gravel")

# Optional interactive version
INTERACTIVE = True
def add_fact_interactive():
    colour = input("Enter new traffic light colour: ").capitalize()
    action = input(f"What action when the light is {colour}? ").upper().replace(" ", "_")
    LIGHTS.add(colour)
    add_rule(f"U_{colour}", 85, lambda f, c=colour: f["Light"] == c, action, 0, "User-added rule.")

if INTERACTIVE:
    add_fact_interactive()

show_kb()

Added rule: R12_white_light -> STOP
Enter new traffic light colour: white
What action when the light is White? stop and reverse
Added rule: U_White -> STOP_AND_REVERSE

--- Current Knowledge Base (rules) ---
[100] R1_obstacle_too_close    -> EMERGENCY_BRAKE        cap= 0 km/h | Obstacle is closer than the distance needed to stop.
[ 95] R2_hitch_unlocked        -> STOP                   cap= 0 km/h | Hitch is not locked; trailer must not move.
[ 90] R3_load_unsecured        -> STOP                   cap= 0 km/h | Load is not secured.
[ 85] R12_white_light          -> STOP                   cap= 0 km/h | White light (e.g. railway crossing signal): stop immediately.
[ 85] U_White                  -> STOP_AND_REVERSE       cap= 0 km/h | User-added rule.
[ 80] R4_red_light             -> STOP                   cap= 0 km/h | Red light: stop.
[ 75] R5_flashing_light        -> STOP_AND_CHECK         cap= 0 km/h | Flashing signal: stop and check the intersection.
[ 70] R6_fog_or_storm          

**Steps 6 & 7: Encode Specific Problem Instance (Facts) & Pose Queries and Get Answers**

In [ ]:
#Step 6: the current observation from sensors, camera and the weather API
current_facts = {
    "HitchLocked": True,
    "LoadSecured": True,
    "Loaded": True,
    "ObstacleDistance": 45.0,   # metres to nearest obstacle ahead (None if clear)
    "Speed": 30.0,              # km/h
    "Light": "Green",
    "Weather": live_weather,    # from Open-Meteo (or fallback)
    "Road": "Paved",
}

#Step 7: the agent queries the knowledge base
def trailer_advice(facts, verbose=True):
    f = derive(facts)
    fired = [r for r in rules if r["if"](f) and not r.get("default")]
    if not fired:                       # default rule applies only when nothing else does
        fired = [r for r in rules if r.get("default")]
    best = max(fired, key=lambda r: r["priority"])
    cap = min(r["cap"] for r in fired)
    result = {"action": best["action"], "speed_cap_kmh": cap,
              "decided_by": best["name"], "reason": best["why"],
              "stopping_distance_m": f["StoppingDistance"],
              "fired": [r["name"] for r in sorted(fired, key=lambda r: -r["priority"])]}
    if verbose:
        print("=== Autonomous Trailer Decision System ===")
        print("Observed:", {k: v for k, v in facts.items()})
        print(f"Derived : Friction={f['Friction']:.2f}, StoppingDistance={f['StoppingDistance']} m")
        print(f"Decision: {result['action']} (speed cap {cap} km/h)")
        print(f"Because : {result['reason']}  [{result['decided_by']}]")
        print("Fired   :", ", ".join(result["fired"]))
    return result

_ = trailer_advice(current_facts)

=== Autonomous Trailer Decision System ===
Observed: {'HitchLocked': True, 'LoadSecured': True, 'Loaded': True, 'ObstacleDistance': 45.0, 'Speed': 30.0, 'Light': 'Green', 'Weather': 'Clear', 'Road': 'Paved'}
Derived : Friction=0.80, StoppingDistance=21.2 m
Decision: PROCEED (speed cap 60 km/h)
Because : Conditions are clear.  [R10_all_clear]
Fired   : R10_all_clear


**Step 8: Querying the knowledge base**

In [ ]:
def scenario(**changes):
    f = dict(current_facts); f.update(changes); return f

queries = {
    "Red light":              scenario(Light="Red"),
    "Flashing light":         scenario(Light="Flashing"),
    "White light (new fact)": scenario(Light="White"),
    "Heavy fog":              scenario(Weather="Fog"),
    "Muddy farm track":       scenario(Road="Muddy", Light="None"),
    "Hitch unlocked":         scenario(HitchLocked=False),
    "Obstacle at 10 m":       scenario(ObstacleDistance=10.0),
    "Rain, loaded, 50 km/h":  scenario(Weather="Rain", Speed=50.0, ObstacleDistance=None),
}
for label, facts in queries.items():
    r = trailer_advice(facts, verbose=False)
    print(f"{label:24} -> {r['action']:22} cap={r['speed_cap_kmh']:>2} km/h  ({r['decided_by']})")

Red light                -> STOP                   cap= 0 km/h  (R4_red_light)
Flashing light           -> STOP_AND_CHECK         cap= 0 km/h  (R5_flashing_light)
White light (new fact)   -> STOP                   cap= 0 km/h  (R12_white_light)
Heavy fog                -> SLOW                   cap=30 km/h  (R6_fog_or_storm)
Muddy farm track         -> SLOW                   cap=20 km/h  (R8_muddy_road)
Hitch unlocked           -> STOP                   cap= 0 km/h  (R2_hitch_unlocked)
Obstacle at 10 m         -> EMERGENCY_BRAKE        cap= 0 km/h  (R1_obstacle_too_close)
Rain, loaded, 50 km/h    -> PROCEED_WITH_CAUTION   cap=40 km/h  (R9_rain_or_dirt)


**Step 9: Debugging the Knowledge Base**

In [ ]:
import itertools

#a. Expected-outcome tests: does the agent do what a human expert would?
tests = [
    (scenario(Light="Red"),                       "STOP"),
    (scenario(HitchLocked=False, Light="Green"),  "STOP"),
    (scenario(LoadSecured=False),                 "STOP"),
    (scenario(ObstacleDistance=5.0),              "EMERGENCY_BRAKE"),
    (scenario(Light="Yellow", ObstacleDistance=None, Weather="Clear"), "SLOW"),
    (scenario(Weather="Storm", ObstacleDistance=None), "SLOW"),
    (scenario(Light="Green", Weather="Clear", ObstacleDistance=None), "PROCEED"),
]
print("--- 9a. Expected-outcome tests ---")
for facts, expected in tests:
    got = trailer_advice(facts, verbose=False)["action"]
    print(("PASS" if got == expected else "FAIL"), f"expected={expected:16} got={got}")

#b. Coverage and conflict check over the whole discrete state space
print("\n--- b. Exhaustive check ---")
unsafe = []
for light, weather, road, hitch, load in itertools.product(
        sorted(LIGHTS), sorted(WEATHER), sorted(ROADS), [True, False], [True, False]):
    f = scenario(Light=light, Weather=weather, Road=road, HitchLocked=hitch,
                 LoadSecured=load, ObstacleDistance=None)
    r = trailer_advice(f, verbose=False)
    #Safety invariant: never PROCEED if hitch/load unsafe or light is Red
    if r["action"] == "PROCEED" and (not hitch or not load or light == "Red"):
        unsafe.append((light, weather, road, hitch, load))
print("States checked :", len(LIGHTS) * len(WEATHER) * len(ROADS) * 4)
print("Safety violations:", len(unsafe))

# Rules with the same priority but different actions would be ambiguous
from collections import defaultdict
by_p = defaultdict(set)
for r in rules: by_p[r["priority"]].add(r["action"])
print("Ambiguous priorities:", [p for p, a in by_p.items() if len(a) > 1] or "none")

#c. Unknown or faulty signals fall back to the default rule (cautious)
print("\n--- 9c. Faulty signal handling ---")
for color in ["Blue", "Teal", "None", "Purple"]:
    try:
        r = trailer_advice(scenario(Light=color, ObstacleDistance=None), verbose=False)
        print(f"Light {color:7} -> {r['action']} ({r['decided_by']})")
    except Exception as e:
        print(f"Light {color:7} -> ERROR {e}")

--- 9a. Expected-outcome tests ---
PASS expected=STOP             got=STOP
PASS expected=STOP             got=STOP
PASS expected=STOP             got=STOP
PASS expected=EMERGENCY_BRAKE  got=EMERGENCY_BRAKE
PASS expected=SLOW             got=SLOW
PASS expected=SLOW             got=SLOW
PASS expected=PROCEED          got=PROCEED

--- b. Exhaustive check ---
States checked : 384
Safety violations: 0
Ambiguous priorities: [85]

--- 9c. Faulty signal handling ---
Light Blue    -> PROCEED_WITH_CAUTION (R11_default)
Light Teal    -> PROCEED_WITH_CAUTION (R11_default)
Light None    -> PROCEED (R10_all_clear)
Light Purple  -> PROCEED_WITH_CAUTION (R11_default)


Known weaknesses found by testing: a light colour such as *Blue* is not in the vocabulary, so only the default rule (R11) fires and the trailer proceeds with caution at 30 km/h.

A stricter design would treat an unrecognised signal as `STOP_AND_CHECK`.

**Step 10: Reflection**

*Challenges*
*   **Rule conflicts.** Several rules can fire at once (e.g. Green light but muddy road in fog). Priorities and "lowest speed cap wins" resolve this, but picking the priority numbers is a judgement call that needs expert review.
*   **External data.** The weather API can be offline, rate-limited or inaccurate for the trailer's exact spot, so the agent needs a safe fallback and should not trust a single reading blindly.
*   **Sensor uncertainty.** The rules treat percepts as certain (hitch locked, distance 45.0 m). Real sensors are noisy and can fail.

*Recommended improvements*
*   Add a safety margin to stopping distance and use the worst of several sensor readings.
*   Treat missing or unrecognised percepts as unsafe (`STOP_AND_CHECK`) rather than defaulting to caution.
*   Add rules for slopes and braking-system faults, trailer sway and speed limits from local regulations.
*   Replace the dictionary rules with a logic engine or add probabilities to handle uncertainty.
